# Artifact-Reject Demo
**HackDuke 2026 — James Mu (Duke) & Derek Mu (CMU)**

This notebook demonstrates the artifact rejection pipeline on synthetic neural signal.
No Synapse device or simulator required — everything runs locally.

### Pipeline
1. Generate synthetic neural-like broadband signal (32 channels, 30 kHz)
2. Inject real-world artifact types: spikes, flatlines, 60 Hz line noise
3. Run **our detector** (MAD + FFT) vs **industry baseline** (fixed amplitude threshold)
4. Compare precision, false positives, and visual output

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
sys.path.insert(0, '../client')

from artifact_reject import (
    detect_mad, detect_60hz, detect_fixed_threshold,
    SAMPLE_RATE_HZ, WINDOW_SAMPLES, N_CHANNELS,
    MAD_THRESHOLD, FIXED_THRESHOLD, SPECTRAL_60HZ_RATIO,
)

plt.style.use('dark_background')
rng = np.random.default_rng(42)
print('Ready.')

## 1. Generate synthetic neural signal

Real neural broadband signal is roughly Gaussian with occasional bursts.
We simulate it as band-limited Gaussian noise (0–6 kHz) to match typical spike-band recordings.

In [ ]:
N_WINDOWS   = 40
DISPLAY_CH  = 0
SIGNAL_STD  = 150   # microvolts, typical spike-band RMS

def generate_clean_signal(n_windows, rng):
    """Generate N windows of synthetic neural broadband signal.
    Shape: (n_windows, N_CHANNELS, WINDOW_SAMPLES)
    """
    return rng.normal(0, SIGNAL_STD, size=(n_windows, N_CHANNELS, WINDOW_SAMPLES)).astype(np.float32)

clean = generate_clean_signal(N_WINDOWS, rng)

# Plot one channel of clean signal
fig, ax = plt.subplots(figsize=(14, 3))
t = np.arange(WINDOW_SAMPLES * 3) / SAMPLE_RATE_HZ * 1000  # ms
ax.plot(t, clean[:3, DISPLAY_CH, :].flatten(), color='#2ecc71', linewidth=0.6)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude (μV)')
ax.set_title(f'Clean synthetic neural signal — channel {DISPLAY_CH} (first 300ms)')
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()
print(f'Signal stats — mean: {clean.mean():.1f} μV  std: {clean.std():.1f} μV')

## 2. Inject artifacts

We inject one of three artifact types into ~30% of windows:
- **Spike** — single saturating transient on one channel (e.g. stimulation artifact, muscle burst)
- **Flatline** — one channel stuck at zero (disconnected/saturated electrode)
- **60 Hz** — sinusoidal line noise added to all channels

In [ ]:
INJECT_PROB      = 0.30
SPIKE_AMPLITUDE  = 2000   # μV — well above neural signal range
NOISE_60HZ_AMP   = 400    # μV

def inject_artifact(window, rng):
    corrupted     = window.copy()
    artifact_type = rng.choice(['spike', 'flatline', '60hz'])

    if artifact_type == 'spike':
        ch  = rng.integers(0, N_CHANNELS)
        idx = rng.integers(0, WINDOW_SAMPLES)
        corrupted[ch, idx] = SPIKE_AMPLITUDE

    elif artifact_type == 'flatline':
        ch = rng.integers(0, N_CHANNELS)
        corrupted[ch, :] = 0.0

    elif artifact_type == '60hz':
        t     = np.arange(WINDOW_SAMPLES) / SAMPLE_RATE_HZ
        noise = NOISE_60HZ_AMP * np.sin(2 * np.pi * 60 * t)
        corrupted += noise[np.newaxis, :]

    return corrupted, artifact_type

# Apply injection
corrupted_windows = []
ground_truth      = []
artifact_types    = []

for w in range(N_WINDOWS):
    if rng.random() < INJECT_PROB:
        win, atype = inject_artifact(clean[w], rng)
        corrupted_windows.append(win)
        ground_truth.append(True)
        artifact_types.append(atype)
    else:
        corrupted_windows.append(clean[w].copy())
        ground_truth.append(False)
        artifact_types.append(None)

n_artifacts = sum(ground_truth)
counts = {t: artifact_types.count(t) for t in ['spike', 'flatline', '60hz']}
print(f'Injected {n_artifacts}/{N_WINDOWS} windows with artifacts')
print(f'  Spikes: {counts["spike"]}  |  Flatlines: {counts["flatline"]}  |  60Hz: {counts["60hz"]}')

## 3. Visualise each artifact type

In [ ]:
def find_first(atype):
    return next(i for i, t in enumerate(artifact_types) if t == atype)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
fig.patch.set_facecolor('#0d1117')
t_ms = np.arange(WINDOW_SAMPLES) / SAMPLE_RATE_HZ * 1000

for ax, atype, color in zip(axes, ['spike', 'flatline', '60hz'], ['#e74c3c', '#e67e22', '#9b59b6']):
    idx = find_first(atype)
    ax.plot(t_ms, clean[idx, DISPLAY_CH], color='#2ecc71', linewidth=0.7, label='clean', alpha=0.7)
    ax.plot(t_ms, corrupted_windows[idx][DISPLAY_CH], color=color, linewidth=0.8, label=f'{atype} artifact')
    ax.set_title(f'{atype.upper()} artifact', color=color, fontsize=12)
    ax.set_xlabel('Time (ms)')
    ax.set_facecolor('#161b22')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Amplitude (μV)')
fig.suptitle('Artifact types — clean (green) vs corrupted (color)', color='white', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Run both detectors across all windows

In [ ]:
ours_results  = []
fixed_results = []

for w in range(N_WINDOWS):
    window = corrupted_windows[w]

    mad_hit,  _  = detect_mad(window)
    hz60_hit, _  = detect_60hz(window)
    ours_hit     = mad_hit or hz60_hit

    fixed_hit, _ = detect_fixed_threshold(window)

    ours_results.append(ours_hit)
    fixed_results.append(fixed_hit)

def score(preds, truth):
    tp = sum(p and t for p, t in zip(preds, truth))
    fp = sum(p and not t for p, t in zip(preds, truth))
    fn = sum(not p and t for p, t in zip(preds, truth))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    recall    = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    return tp, fp, fn, precision, recall

ours_tp,  ours_fp,  ours_fn,  ours_prec,  ours_rec  = score(ours_results,  ground_truth)
fixed_tp, fixed_fp, fixed_fn, fixed_prec, fixed_rec = score(fixed_results, ground_truth)

print(f'{'Detector':<22} {'TP':>6} {'FP':>6} {'FN':>6} {'Precision':>10} {'Recall':>8}')
print('-' * 60)
print(f'{'Ours (MAD+FFT)':<22} {ours_tp:>6} {ours_fp:>6} {ours_fn:>6} {ours_prec:>10.1%} {ours_rec:>8.1%}')
print(f'{'Fixed threshold':<22} {fixed_tp:>6} {fixed_fp:>6} {fixed_fn:>6} {fixed_prec:>10.1%} {fixed_rec:>8.1%}')

## 5. Visual comparison — cleaned signal output

In [ ]:
# Build scrolling signal for all windows
raw_signal   = np.concatenate([corrupted_windows[w][DISPLAY_CH] for w in range(N_WINDOWS)])
ours_cleaned = np.concatenate([
    np.zeros(WINDOW_SAMPLES) if ours_results[w] else corrupted_windows[w][DISPLAY_CH]
    for w in range(N_WINDOWS)
])
fixed_cleaned = np.concatenate([
    np.zeros(WINDOW_SAMPLES) if fixed_results[w] else corrupted_windows[w][DISPLAY_CH]
    for w in range(N_WINDOWS)
])

t_s = np.arange(len(raw_signal)) / SAMPLE_RATE_HZ * 1000  # ms

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Artifact Rejection — Full Signal View', color='white', fontsize=13, fontweight='bold')

panels = [
    (raw_signal,    '#ecf0f1', 'Raw Signal (with injected artifacts)'),
    (ours_cleaned,  '#2ecc71', f'Ours (MAD+FFT)  |  Precision={ours_prec:.0%}  Recall={ours_rec:.0%}  FP={ours_fp}'),
    (fixed_cleaned, '#e67e22', f'Fixed Threshold  |  Precision={fixed_prec:.0%}  Recall={fixed_rec:.0%}  FP={fixed_fp}'),
]

for ax, (signal, color, title) in zip(axes, panels):
    ax.plot(t_s, signal, color=color, linewidth=0.6)
    ax.set_title(title, color=color, fontsize=10, pad=4)
    ax.set_facecolor('#161b22')
    ax.set_ylabel('μV')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

    # Shade artifact windows
    for w in range(N_WINDOWS):
        if ground_truth[w]:
            x0 = w * WINDOW_SAMPLES / SAMPLE_RATE_HZ * 1000
            x1 = x0 + WINDOW_SAMPLES / SAMPLE_RATE_HZ * 1000
            ax.axvspan(x0, x1, color='#e74c3c', alpha=0.15)

axes[-1].set_xlabel('Time (ms)')

# Legend
patch = mpatches.Patch(color='#e74c3c', alpha=0.4, label='Artifact window (ground truth)')
axes[0].legend(handles=[patch], loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Summary

The key takeaway is visible in panel 2 vs panel 3:
- **Our method** suppresses only artifact windows, preserving the clean signal in between
- **Fixed threshold** over-suppresses — blanking large portions of clean signal because it can't adapt to signal amplitude

This over-suppression is the real cost in a clinical or BCI context: it throws away good neural data, degrading decoder performance even when there are no artifacts.

In [ ]:
ours_blanked  = ours_results.count(True)
fixed_blanked = fixed_results.count(True)
real_artifacts = sum(ground_truth)

print(f'Windows blanked by our method:       {ours_blanked}/{N_WINDOWS}  ({ours_blanked/N_WINDOWS:.0%})')
print(f'Windows blanked by fixed threshold:  {fixed_blanked}/{N_WINDOWS}  ({fixed_blanked/N_WINDOWS:.0%})')
print(f'Actual artifact windows:             {real_artifacts}/{N_WINDOWS}  ({real_artifacts/N_WINDOWS:.0%})')
print()
print(f'Clean signal preserved by ours:      {(N_WINDOWS - ours_blanked)/N_WINDOWS:.0%}')
print(f'Clean signal preserved by fixed:     {(N_WINDOWS - fixed_blanked)/N_WINDOWS:.0%}')